In [2]:
import pyspark
from pyspark.sql import SparkSession

In [3]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/07 21:18:03 WARN Utils: Your hostname, minh-HP-250-G8-Notebook-PC, resolves to a loopback address: 127.0.1.1; using 192.168.1.12 instead (on interface wlo1)
26/03/07 21:18:03 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/07 21:18:04 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
!wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet

--2026-03-07 20:13:12--  https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 2600:9000:21c8:2200:b:20a5:b140:21, 2600:9000:21c8:d800:b:20a5:b140:21, 2600:9000:21c8:5c00:b:20a5:b140:21, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|2600:9000:21c8:2200:b:20a5:b140:21|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 71134255 (68M) [binary/octet-stream]
Saving to: ‘yellow_tripdata_2025-11.parquet’

yellow_tripdata_202 100%[===================>]  67.84M  12.8MB/s    in 6.0s    

2026-03-07 20:13:18 (11.3 MB/s) - ‘yellow_tripdata_2025-11.parquet’ saved [71134255/71134255]



In [6]:
df_yellow = spark.read \
    .parquet("yellow_tripdata_2025-11.parquet")

In [7]:
df_yellow.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       7| 2025-11-01 00:13:25|  2025-11-01 00:13:25|              1|         1.68|         1|                 N|          43|    

In [6]:
df_yellow.repartition(4).write.parquet('yellow_tripdata_partitioned')

In [9]:
df_yellow.createOrReplaceTempView('yellow_data')

In [10]:
result_3 = spark.sql("""
    SELECT 
        COUNT(1)
    FROM
        yellow_data
    WHERE
        TO_DATE(tpep_pickup_datetime) = "2025-11-15"
""")

In [11]:
result_3.show()

+--------+
|count(1)|
+--------+
|  162604|
+--------+



In [12]:
result_4 = spark.sql("""
    SELECT 
        (unix_timestamp(tpep_dropoff_datetime) - unix_timestamp(tpep_pickup_datetime)) / 3600 AS duration_hours
    FROM 
        yellow_data
    ORDER BY 
        duration_hours DESC
    LIMIT 1
""")

In [13]:
result_4.show()

+-----------------+
|   duration_hours|
+-----------------+
|90.64666666666666|
+-----------------+



In [19]:
!wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

--2026-03-07 20:37:28--  https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 2600:9000:21c8:e400:b:20a5:b140:21, 2600:9000:21c8:d600:b:20a5:b140:21, 2600:9000:21c8:4600:b:20a5:b140:21, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|2600:9000:21c8:e400:b:20a5:b140:21|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12331 (12K) [text/csv]
Saving to: ‘taxi_zone_lookup.csv’

taxi_zone_lookup.cs 100%[===================>]  12.04K  --.-KB/s    in 0.002s  

2026-03-07 20:37:28 (7.24 MB/s) - ‘taxi_zone_lookup.csv’ saved [12331/12331]



In [20]:
df_zones = spark.read \
    .option("header", "true") \
    .csv("taxi_zone_lookup.csv")

In [21]:
df_zones.show()

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
|         6|Staten Island|Arrochar/Fort Wad...|   Boro Zone|
|         7|       Queens|             Astoria|   Boro Zone|
|         8|       Queens|        Astoria Park|   Boro Zone|
|         9|       Queens|          Auburndale|   Boro Zone|
|        10|       Queens|        Baisley Park|   Boro Zone|
|        11|     Brooklyn|          Bath Beach|   Boro Zone|
|        12|    Manhattan|        Battery Park| Yellow Zone|
|        13|    Manhattan|   Battery Park City| Yellow Zone|
|        14|     Brookly

In [24]:
df_yellow_join = df_yellow.join(df_zones, (df_yellow.PULocationID) == (df_zones.LocationID))

In [25]:
df_yellow_join.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+----------+---------+--------------------+------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|LocationID|  Borough|                Zone|service_zone|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+----------+---------+

In [26]:
df_yellow_join.createOrReplaceTempView('yellow_join')

In [41]:
result_6 = spark.sql("""
    SELECT 
        Zone,
        COUNT(1) AS frequent_pickup
    FROM
        yellow_join
    WHERE
        MONTH(TO_DATE(tpep_pickup_datetime)) = '11'
    GROUP BY
        Zone
    ORDER BY frequent_pickup ASC
    LIMIT 3
""")

In [43]:
result_6.head(3)

[Row(Zone="Governor's Island/Ellis Island/Liberty Island", frequent_pickup=1),
 Row(Zone='Arden Heights', frequent_pickup=1),
 Row(Zone="Eltingville/Annadale/Prince's Bay", frequent_pickup=1)]